### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>
- uses OSMNx, pandas, geopandas, rasterio
- may include content from ChatGPT or GitHub Copilot

### Features
Check the quality of the model inputs for the TI-City Model:
- Value ranges
- Coregistration
- Normalisation

### Prerequisites
- AOI vector file (i.e., shapefile or geopackage)
- Raster reference file with the same extent as the AOI

## Roadmap:

### Version 1.2
1. [X] Add the 'exposed.asc' layer

### Version 1.1
1. [X] Update file and folder locations to Uni Bonn file structure
2. Update JAK data to GHSL (for 2015-2025) for  Built up & Distance to built up
   - [X] discrete_list
   - [X] distance_dict
   - [X] continuous_list
   - [X] renaming_dict
3. Rename date-dependent file names. From "2000" to "early" and "2015" to "late", retaining two calibration years, but making them flexible for other specifications.
   - [X] discrete_list
   - [X] distance_dict
   - [X] continuous_list
   - [X] renaming_dict

### Version 1.0
1. [X] Implement negative values check
2. [X] Implement coregister modulo
3. [X] Convert TIFF to ASCII
4. [X] Re-name and copy to TI-City model folder

# imports

In [ ]:
#%%time
# utils
import os
from pathlib import Path
import sys
import fnmatch
import shutil
from logging import error

# maths
import numpy as np
from scipy.ndimage import distance_transform_edt #, gaussian_filter


# GIS
import geopandas as gpd

# raster stuff
import rasterio

from rasterio.warp import Resampling #reproject, calculate_default_transform,
from notebooks.ti_city_00_helper_functions import *


# init

In [ ]:
#%%time

case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_ascii_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]
    simulation_years = [2025, 2050]

##%run ./ti_city_00_raster_functions.ipynb

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn


# 1. check input raster quality
- negative values
- dimensions

## Quick-and-dirty conversion of distance-function raster to utility

In [ ]:
## need to integrate this into the rasterise functions creating the rasters

#%run ./ti_city_00_raster_functions.ipynb

distance_dict = {'airport.asc': case_city + '_LOC_airports_osm_2025_distance_normal_150m.tif', 
                 'cbd.asc': case_city + '_LOC_cbds_osm_2025_distance_normal_150m.tif',
                 'healthfacilities.asc': case_city + '_LOC_health_osm_2025_distance_normal_150m.tif', 
                 'markets.asc': case_city + '_LOC_markets_osm_2025_distance_normal_150m.tif', 
                 'shoppingmalls.asc': case_city + '_LOC_malls_osm_2025_distance_normal_150m.tif', 
                 'schools.asc': case_city + '_LOC_schools_osm_2025_distance_normal_150m.tif', 
                 'suburban.asc': case_city + '_LOC_suburban_centres_osm_2025_distance_normal_150m.tif', 
                 'attractive_early.asc': case_city + '_LOC_attractive_areas_osm_2025_distance_normal_150m.tif', # APS: I need to replace this file with the actual 2000 data
                 'attractive_late.asc': case_city + '_LOC_attractive_areas_osm_2025_distance_normal_150m.tif',
                 'water.asc': case_city + '_HYD_water_distance_normal_150m.tif',
                 'roads.asc': case_city + '_TRA_roads_osm_2025_distance_normal_150m.tif',
                 'density_early.asc':case_city + '_POP_density_normal_2000_WDPop_150m.tif',
                 'density_late.asc':case_city + '_POP_density_normal_2020_WDPop_150m.tif',
                 'bu_early_dist.asc': case_city + '_URB_urbanisation_2015_GHSL_distance_normal_150m.tif',
                 'bu_late_dist.asc': case_city + '_URB_urbanisation_2025_GHSL_distance_normal_150m.tif',
                 'exposed.asc': case_city + '_HYD_flood_hazard_150m.tif'
                 }

no_file_list = []
ut_raster_dict = {}

for ascii, tiff in distance_dict.items():
   distance_url = interim_path / tiff
   utility_raster = None
   utility_file_name = '' 
   #if '_distance_normal_' in tiff:
   utility_file_name = tiff.replace('_distance_normal_', '_utility_')
   
   #elif '_URB_distance_' in tiff:
   #   utility_file_name = tiff.replace('_distance_urbanisation_', '_utility_urbanisation_')
   #   print('*** processing ',utility_file_name)
   # check if the file already exists in the model inputs folder
   if os.path.exists(interim_path / utility_file_name):
       print('*** file already exists found:', utility_file_name)
   elif (ascii.split('.')[0] == 'density_early') | (ascii.split('.')[0] == 'density_late'):
      utility_file_name = tiff.replace('_normal_', '_utility_')  
      print('\n*** processing ',utility_file_name)
      utility_raster = convert_distance_to_utility(distance_url, density_boolean=True)    
   else:
      print('\n*** processing ',utility_file_name)
      utility_raster = convert_distance_to_utility(distance_url, density_boolean=False)
   # export the raster:
   outfile = interim_path / utility_file_name
      
   if utility_raster is not None:
      print(ascii, utility_raster.shape, 'min:', np.nanmin(utility_raster), 'max:', np.nanmax(utility_raster))
      try:
         export_raster(raster=utility_raster,
                        url=outfile,
                        data_type=rasterio.float32,
                        ref_raster_path=ref_raster_path
                        )
      except Exception as e:
         print('*** error exporting utility raster for:', ascii, '\nerror:', e)            
   else:
       no_file_list.append(ascii)
       print('*** no utility raster created for:', ascii)
       continue
   

   print('Raster exported successfully:', utility_file_name)
#print('create stand-in rasters for:',no_file_list)  


*** file already exists found: JAK_LOC_airports_osm_2025_utility_150m.tif
*** no utility raster created for: airport.asc
*** file already exists found: JAK_LOC_cbds_osm_2025_utility_150m.tif
*** no utility raster created for: cbd.asc
*** file already exists found: JAK_LOC_health_osm_2025_utility_150m.tif
*** no utility raster created for: healthfacilities.asc
*** file already exists found: JAK_LOC_markets_osm_2025_utility_150m.tif
*** no utility raster created for: markets.asc
*** file already exists found: JAK_LOC_malls_osm_2025_utility_150m.tif
*** no utility raster created for: shoppingmalls.asc
*** file already exists found: JAK_LOC_schools_osm_2025_utility_150m.tif
*** no utility raster created for: schools.asc
*** file already exists found: JAK_LOC_suburban_centres_osm_2025_utility_150m.tif
*** no utility raster created for: suburban.asc
*** file already exists found: JAK_LOC_attractive_areas_osm_2025_utility_150m.tif
*** no utility raster created for: attractive_early.asc
*** fi

## Check for negative values

In [6]:
# final check of all the model input files
discrete_list = []
continuous_list = []
slope_list = []
discrete_list  = [case_city + '_ECO_real_estate_reclass_150m.tif',
                  case_city + '_LIM_districts_GADM4_150m.tif',
                  case_city + '_LIM_exclusion_layer_150m.tif',
                  #case_city + '_POP_census_income_150m.tif',
                  case_city + '_TRA_roads_OSM_2025_major_150m.tif',
                  #case_city + '_URB_tenure_150m.tif',
                  #case_city + '_URB_urbanisation_1985_EOC_WUF_150m.tif',
                  case_city + '_URB_urbanisation_2015_GHSL_150m.tif',
                  case_city + '_URB_urbanisation_2025_GHSL_150m.tif',
                  case_city + '_LIM_non_residential_areas_150m.tif',
                  case_city + '_HYD_flood_hazard_150m.tif']
continuous_list = [case_city + '_LOC_airports_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_airports_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_attractive_areas_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_attractive_areas_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_CBDs_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_CBDs_OSM_2024_utility_150m.tif', 
                   case_city + '_LOC_health_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_health_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_malls_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_malls_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_markets_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_markets_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_schools_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_schools_OSM_2024_utility_150m.tif',
                   case_city + '_LOC_suburban_centres_OSM_2025_utility_150m.tif', # APS: until 06/03/2025 '_LOC_suburban_centres_OSM_2024_utility_150m.tif',
                   case_city + '_POP_density_normal_2000_WDPop_150m.tif',
                   case_city + '_POP_density_normal_2020_WDPop_150m.tif',
                   case_city + '_TRA_roads_OSM_2025_utility_150m.tif',
                   case_city + '_URB_urbanisation_2015_GHSL_utility_150m.tif',
                   case_city + '_URB_urbanisation_2025_GHSL_utility_150m.tif',
                   case_city + '_HYD_water_utility_150m.tif']

slope_list=[case_city + '_DEM_slope_pct_150m.tif']

raster_list = [raster for raster in os.listdir(model_inputs_150m_path) if raster.endswith('.tif')]
for r in raster_list:
    print(r)

JAK_DEM_slope_pct_150m.tif
JAK_DEM_slope_pct_TanDEM_X_150m.tif
JAK_ECO_real_estate_reclass_150m.tif
JAK_HYD_water_utility_150m.tif
JAK_LIM_districts_GADM4_150m.tif
JAK_LIM_exclusion_layer_150m.tif
JAK_LIM_non_residential_areas_150m.tif
JAK_LOC_airports_OSM_2025_utility_150m.tif
JAK_LOC_attractive_areas_OSM_2025_utility_150m.tif
JAK_LOC_CBDs_OSM_2025_utility_150m.tif
JAK_LOC_health_OSM_2025_utility_150m.tif
JAK_LOC_malls_OSM_2025_utility_150m.tif
JAK_LOC_markets_OSM_2025_utility_150m.tif
JAK_LOC_schools_OSM_2025_utility_150m.tif
JAK_LOC_suburban_centres_OSM_2025_utility_150m.tif
JAK_POP_density_normal_2000_WDPop_150m.tif
JAK_POP_density_normal_2015_GHSL_150m.tif
JAK_POP_density_normal_2020_WDPop_150m.tif
JAK_POP_density_normal_2025_GHSL_150m.tif
JAK_TRA_roads_OSM_2024_all_150m.tif
JAK_TRA_roads_OSM_2025_major_150m.tif
JAK_TRA_roads_OSM_2025_utility_150m.tif
JAK_URB_urbanisation_2015_GHSL_150m.tif
JAK_URB_urbanisation_2015_GHSL_150m_distance_normal.tif
JAK_URB_urbanisation_2015_GHSL_util

In [7]:
# copy the rasters from the interim_path that are missing in the model_inputs_150 folder
set_replace_file = True
print('*** checking the discrete list:')
for file in discrete_list:
    if file in raster_list:
        
        if set_replace_file == True:
            origin_path = interim_path / file
            destination_path = model_inputs_150m_path / file
            origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
            destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
            if os.path.exists(origin_path):
                shutil.copy(origin_path, destination_path)
                print('replaced', str(destination_name))
            else:
                print('*** file not found:', str(origin_name))
        else:
            print(file, 'is already in the model inputs folder')
    else:
        origin_path = interim_path / file
        destination_path = model_inputs_150m_path / file
        origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
        destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
        if os.path.exists(origin_path):
            shutil.copy(origin_path, destination_path)
            print('copied', str(destination_name))
        else:
            
            print('*** file not found:', str(origin_name))
print('\n*** checking the continuous list:')
for file in continuous_list:
    
    if file in raster_list:
        
        if set_replace_file == True:
            origin_path = interim_path / file
            destination_path = model_inputs_150m_path / file
            origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
            destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
            if os.path.exists(origin_path):
                shutil.copy(origin_path, destination_path)
                print('replaced', str(destination_name))
            else:
                print('*** file not found:', str(origin_name))
        else:
            print(file, 'is already in the model inputs folder')
    else:
        origin_path = interim_path / file
        destination_path = model_inputs_150m_path / file
        origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
        destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
        if os.path.exists(origin_path):
            shutil.copy(origin_path, destination_path)
            print('copied', str(destination_name))
        else:
            print('*** file not found:', str(origin_name))
print('\n*** checking the slope list:')
for file in slope_list:
    
    if file in raster_list:
        
        if set_replace_file == True:
            origin_path = interim_path / file
            destination_path = model_inputs_150m_path / file
            origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
            destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
            if os.path.exists(origin_path):
                shutil.copy(origin_path, destination_path)
                print('replaced', str(destination_name))
            else:
                print('*** file not found:', str(origin_name))
        else:
            print(file, 'is already in the model inputs folder')
    else:
        origin_path = interim_path / file
        destination_path = model_inputs_150m_path / file
        origin_name = str(origin_path).split('\\')[-2] + '\\' + str(origin_path).split('\\')[-1]
        destination_name = str(destination_path).split('\\')[-2] + '\\' + str(destination_path).split('\\')[-1]
        if os.path.exists(origin_path):
            shutil.copy(origin_path, destination_path)
            print('copied', str(destination_name))
        else:
            print('*** file not found:', str(origin_name))

*** checking the discrete list:
replaced TIFF_150m\JAK_ECO_real_estate_reclass_150m.tif
replaced TIFF_150m\JAK_LIM_districts_GADM4_150m.tif
replaced TIFF_150m\JAK_LIM_exclusion_layer_150m.tif
replaced TIFF_150m\JAK_TRA_roads_OSM_2025_major_150m.tif
replaced TIFF_150m\JAK_URB_urbanisation_2015_GHSL_150m.tif
replaced TIFF_150m\JAK_URB_urbanisation_2025_GHSL_150m.tif
replaced TIFF_150m\JAK_LIM_non_residential_areas_150m.tif
copied TIFF_150m\JAK_HYD_flood_hazard_150m.tif

*** checking the continuous list:
replaced TIFF_150m\JAK_LOC_airports_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_attractive_areas_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_CBDs_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_health_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_malls_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_markets_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_schools_OSM_2025_utility_150m.tif
replaced TIFF_150m\JAK_LOC_suburban_centres_OSM_2025_utility_1

In [9]:
negative_list = []
print('checking the Discrete list')
for r in discrete_list:    
    with rasterio.open(model_inputs_150m_path/r) as raster:
        array = raster.read(1)
        if np.nanmin(array) < 0:
            print(r, 'has negative values')
            array = np.where(array < 0, 0, array)
            negative_list.append(r)
print('checking the Slope list')
for r in slope_list:
    with rasterio.open(model_inputs_150m_path/r) as raster:
        array = raster.read(1)
        if np.nanmin(array) < 0:
            print(r, 'has negative values')
            array = np.where(array < 0, 0, array)
            negative_list.append(r)
# check for nan values as -9999 and convert them to 0
print('checking the Discrete list')
for r in discrete_list:
    with rasterio.open(model_inputs_150m_path/r) as raster:
        array = raster.read(1)
        if np.isnan(array).any():
            print(r, 'Has NaN as -9999')
            array = np.where(np.nanmin(array) == -9999, 0, array)
            negative_list.append(r)
print('rasters to remove negative values from:', negative_list)

checking the Discrete list
checking the Slope list
checking the Discrete list
rasters to remove negative values from: []


### correct negative values

In [10]:
#%run ./ti_city_00_raster_functions.ipynb
if len(negative_list) > 0:
    print('checking the following rasters for negative values:', negative_list)
    for raster in negative_list: 
        check_negative_raster(
            r = raster, 
            path=model_inputs_150m_path,
            ref_raster_path = ref_raster_150_path)
else:   
    print('no rasters with negative values found')

no rasters with negative values found


## Normalizing continuous values

In [11]:
for r in continuous_list:
    with rasterio.open(model_inputs_150m_path/r, 'r') as raster:
        array = raster.read(1)
        try:
            array = np.where(np.isnan(array), 0, array)
            array = np.where(array < 0, 0, array)
            array = (array - np.nanmin(array))/(np.nanmax(array) - np.nanmin(array))
            array = np.round(array,3)
            #[0 if i < 0 else i for i in array]
            array = np.where(array < 0, 0, array)
        except Exception as e:
            print('error in raster', r, '\nerror',e)
        #plot_continuous_raster(array, r)
        # export the normalised array as a raster
        new_name = str(r.split('.')[0] + '_new_150m.tif')
        export_geotiff(raster = array,
                    out_transform = raster.transform, 
                    out_meta = raster.meta,
                    export_path = model_inputs_150m_path/new_name,
                    data_type=rasterio.float64)
        # rename the existing raster with '_old"
    try:
        # move the old raster to the '_old' folder and rename the new raster
        old_folder = model_inputs_150m_path / '_old'
        old_folder.mkdir(exist_ok=True)
        old_raster = old_folder/r
        if os.path.exists(old_raster): os.remove(old_raster)
        path_as_string = str(model_inputs_150m_path / r)
        shutil.move(model_inputs_150m_path/r, old_folder/r)

        new_raster_old_name = model_inputs_150m_path / new_name
        new_raster_new_name = Path(path_as_string)
        new_raster_old_name.rename(new_raster_new_name)
      
    except Exception as e:
        print('error in raster', r, '\nerror',e)
print('done')

C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_airports_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_attractive_areas_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_CBDs_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_health_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_malls_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_markets_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_schools_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_LOC_suburban_centres_OSM_2025_utility_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_POP_density_normal_2000_WDPop_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\TIFF_150m\JAK_POP_density_normal_2020_WDPop_150m_new_150m.tif
C:\Sciebo\05_GIS\JAK\model_inputs\T

## Co-register to the AOI

In [12]:
# check the raster dimensions to make sure they are the same
raster_list = [raster for raster in os.listdir(model_inputs_150m_path) if raster.endswith('.tif')]
reproject_raster_list = []

for raster in raster_list:
    try:
        with rasterio.open(model_inputs_150m_path/raster, 'r') as r:
            
            with rasterio.open(ref_raster_150_path, 'r') as ref_raster:
                if r.width != ref_raster.width:
                    print('***',raster, 'has a different **width** from the reference with',r.width, '(versus',ref_raster.width, 'in the reference).')
                    reproject_raster_list.append(raster)
                    if r.height != ref_raster.height:
                        print('***',raster, 'has a different **height** from the reference with',r.height, '(versus',ref_raster.height, 'in the reference).')
                elif r.height != ref_raster.height:
                    print('***',raster, 'has a different **height** from the reference with',r.height, '(versus',ref_raster.height, 'in the reference).')
                    reproject_raster_list.append(raster)
                else: print(raster, 'dimensions are good.')
    except Exception as e:
        print('*** error in raster', raster, '\nerror',e)
print('Rasters to reproject:', reproject_raster_list)     
        

JAK_DEM_slope_pct_150m.tif dimensions are good.
JAK_DEM_slope_pct_TanDEM_X_150m.tif dimensions are good.
JAK_ECO_real_estate_reclass_150m.tif dimensions are good.
JAK_HYD_flood_hazard_150m.tif dimensions are good.
JAK_HYD_water_utility_150m.tif dimensions are good.
JAK_LIM_districts_GADM4_150m.tif dimensions are good.
JAK_LIM_exclusion_layer_150m.tif dimensions are good.
JAK_LIM_non_residential_areas_150m.tif dimensions are good.
JAK_LOC_airports_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_attractive_areas_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_CBDs_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_health_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_malls_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_markets_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_schools_OSM_2025_utility_150m.tif dimensions are good.
JAK_LOC_suburban_centres_OSM_2025_utility_150m.tif dimensions are good.
JAK_POP_density_normal_2000_WDPop_150m.tif dimens

In [13]:
# if needed, co-register the raster to the reference raster
for r in reproject_raster_list:
    new_name = str(r.split('.')[0] + '_new_150m.tif')
    outfile = model_inputs_150m_path / new_name

    if raster in set(continuous_list + slope_list):
        reproj_match(infile = model_inputs_150m_path/raster, 
                 match = ref_raster_150_path, 
                 outfile = outfile, 
                 resampling=Resampling.bilinear)
    elif raster in set(discrete_list):
        reproj_match(infile = model_inputs_150m_path/raster, 
                    match = ref_raster_150_path, 
                    outfile = outfile, 
                    resampling=Resampling.nearest)
    try:
        # move the old raster to the '_old' folder and rename the new raster
        old_folder = model_inputs_150m_path / '_old'
        old_folder.mkdir(exist_ok=True)
        old_raster = old_folder/r
        if os.path.exists(old_raster): os.remove(old_raster)
        path_as_string = str(model_inputs_150m_path / r)
        shutil.move(model_inputs_150m_path/r, old_folder/r)

        new_raster_old_name = model_inputs_150m_path / new_name
        new_raster_new_name = Path(path_as_string)
        new_raster_old_name.rename(new_raster_new_name)
      
    except Exception as e:
        print(e)

# 2. export ASCII files to the model_inputs ASCII folder

In [14]:
raster_list = [raster for raster in os.listdir(model_inputs_150m_path) if raster.endswith('.tif')]
#for r in raster_list:
#    print(r)

In [15]:
# check again if the rasters are in the folder
# copy the rasters from the interim_path that are missing in the model_inputs_150 folder
print('*** checking the discrete list:')
for file in discrete_list:
    if file in raster_list:
        print(file, 'is already in the model inputs folder')
    else:
        print('*** file not found:', str(origin_name))
print('\n*** checking the continuous list:')
for file in continuous_list:
    
    if file in raster_list:
        print(file, 'is already in the model inputs folder')
    else:
        print('*** file not found:', str(origin_name))
print('\n*** checking the slope list:')
for file in slope_list:
    
    if file in raster_list:
        print(file, 'is already in the model inputs folder')
    else:
        print('*** file not found:', str(origin_name))

*** checking the discrete list:
JAK_ECO_real_estate_reclass_150m.tif is already in the model inputs folder
JAK_LIM_districts_GADM4_150m.tif is already in the model inputs folder
JAK_LIM_exclusion_layer_150m.tif is already in the model inputs folder
JAK_TRA_roads_OSM_2025_major_150m.tif is already in the model inputs folder
JAK_URB_urbanisation_2015_GHSL_150m.tif is already in the model inputs folder
JAK_URB_urbanisation_2025_GHSL_150m.tif is already in the model inputs folder
JAK_LIM_non_residential_areas_150m.tif is already in the model inputs folder
JAK_HYD_flood_hazard_150m.tif is already in the model inputs folder

*** checking the continuous list:
JAK_LOC_airports_OSM_2025_utility_150m.tif is already in the model inputs folder
JAK_LOC_attractive_areas_OSM_2025_utility_150m.tif is already in the model inputs folder
JAK_LOC_CBDs_OSM_2025_utility_150m.tif is already in the model inputs folder
JAK_LOC_health_OSM_2025_utility_150m.tif is already in the model inputs folder
JAK_LOC_malls

In [16]:
#%run ./ti_city_00_raster_functions.ipynb
for raster in raster_list: 
    file_name = str(raster.replace('.tif', '.asc'))
    export_path = ascii150_path / file_name
    try:
        with rasterio.open(model_inputs_150m_path/raster, 'r') as r:
            export_raster_to_ascii(raster_obj = r.read(1),                             
                                export_path = export_path, 
                                ref_raster_path = ref_raster_150_path,
                                raster_data_type=r.dtypes[0])
    except Exception as e:
        print(e)
    print('exported', str(export_path).split('\\')[-2:]) #.split('\\')[-2:]
print('done')

exported ['ASCII_150m', 'JAK_DEM_slope_pct_150m.asc']
exported ['ASCII_150m', 'JAK_DEM_slope_pct_TanDEM_X_150m.asc']
exported ['ASCII_150m', 'JAK_ECO_real_estate_reclass_150m.asc']
exported ['ASCII_150m', 'JAK_HYD_flood_hazard_150m.asc']
exported ['ASCII_150m', 'JAK_HYD_water_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LIM_districts_GADM4_150m.asc']
exported ['ASCII_150m', 'JAK_LIM_exclusion_layer_150m.asc']
exported ['ASCII_150m', 'JAK_LIM_non_residential_areas_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_airports_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_attractive_areas_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_CBDs_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_health_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_malls_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_markets_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LOC_schools_OSM_2025_utility_150m.asc']
exported ['ASCII_150m', 'JAK_LO

# 3. Rename the ASCII files and copy them to the git folder
matches the TI-City model specifications

In [17]:
print(ti_city_ascii_path)
# list all the files in the working dir

ti_city_files = []
for file in os.listdir(ti_city_ascii_path):
    if fnmatch.fnmatch(file, '*.asc'):
        ti_city_files.append(file)
ti_city_files

..\model\JAK\data\in


['airport.asc',
 'attractive_early.asc',
 'attractive_late.asc',
 'bu_early_dist.asc',
 'bu_late_dist.asc',
 'cbd.asc',
 'density_early.asc',
 'density_late.asc',
 'districts.asc',
 'exclusion.asc',
 'exclusion_ssp1.asc',
 'exclusion_ssp2.asc',
 'exclusion_ssp3.asc',
 'healthfacilities.asc',
 'landvalue.asc',
 'markets.asc',
 'road-presence.asc',
 'roads.asc',
 'schools.asc',
 'shoppingmalls.asc',
 'slope.asc',
 'structureplan.asc',
 'suburban.asc',
 'urban_early.asc',
 'urban_late.asc',
 'water.asc']

In [ ]:
# build a renaming dictionary between the TI-City ascii names and the input ascii names
renaming_dict = {'slope.asc': case_city + '_DEM_slope_pct_150m.asc', 
                 'districts.asc': case_city + '_lim_districts_gadm4_150m.asc',
                 'structureplan.asc': case_city + '_LIM_non_residential_areas_150m.asc', # APS: I need to replace this file with the actual masterplan
                 #'income.asc': case_city + '_pop_census_income_150m.asc',  # APS 18.11.2024 Need to estimate for Manila
                 'airport.asc': case_city + '_loc_airports_osm_2025_utility_150m.asc', 
                 'cbd.asc': case_city + '_loc_cbds_osm_2025_utility_150m.asc',
                 'healthfacilities.asc': case_city + '_loc_health_osm_2025_utility_150m.asc', 
                 'markets.asc': case_city + '_loc_markets_osm_2025_utility_150m.asc', 
                 'shoppingmalls.asc': case_city + '_loc_malls_osm_2025_utility_150m.asc', 
                 'schools.asc': case_city + '_loc_schools_osm_2025_utility_150m.asc', 
                 'suburban.asc': case_city + '_loc_suburban_centres_osm_2025_utility_150m.asc', 
                 'attractive_early.asc': case_city + '_loc_attractive_areas_osm_2025_utility_150m.asc', # APS: I need to replace this file with the actual 2000 data
                 'attractive_late.asc': case_city + '_loc_attractive_areas_osm_2025_utility_150m.asc', 
                 'water.asc': case_city + '_hyd_water_utility_150m.asc',
                 'roads.asc': case_city + '_TRA_roads_OSM_2025_utility_150m.asc', 
                 #
                 'bu_early_dist.asc': case_city + '_URB_urbanisation_2015_GHSL_utility_150m.asc', 
                 'bu_late_dist.asc': case_city + '_URB_urbanisation_2025_GHSL_utility_150m.asc', 
                 #'tenure.asc': case_city + '_urb_tenure_150m.asc', 
                 'density_early.asc': case_city + '_POP_density_normal_2000_WDPop_150m.asc',
                 'density_late.asc': case_city + '_POP_density_normal_2020_WDPop_150m.asc', 
                 'landvalue.asc': case_city + '_ECO_real_estate_reclass_150m.asc',
                 'road-presence.asc': case_city + '_TRA_roads_OSM_2025_major_150m.asc', 
                 'urban_early.asc': case_city + '_URB_urbanisation_2015_GHSL_150m.asc', 
                 'urban_late.asc': case_city + '_URB_urbanisation_2025_GHSL_150m.asc',
                 'exclusion_s1.asc': case_city + '_lim_exclusion_layer_150m.asc',
                 'exclusion_s2.asc': case_city + '_lim_exclusion_layer_150m.asc',
                 'exclusion_s3.asc': case_city + '_lim_exclusion_layer_150m.asc',
                 'exclusion_s4.asc': case_city + '_lim_exclusion_layer_150m.asc', 
                 'exposed.asc': case_city + '_HYD_flood_hazard_150m.asc'
}
# rasters not included in the above correpondence (yet): income, tenure
# # files as 'stand-ins': structureplan, attractive_early

print('The destination folder is: ',ti_city_ascii_path)
print('The renaming dictionary has', len(renaming_dict), 'entries\n')
for i in renaming_dict.values():
    print(i)


The destination folder is:  ..\model\JAK\data\in
The renaming dictionary has 27 entries

JAK_DEM_slope_pct_150m.asc
JAK_lim_exclusion_layer_150m.asc
JAK_lim_districts_gadm4_150m.asc
JAK_LIM_non_residential_areas_150m.asc
JAK_loc_airports_osm_2025_utility_150m.asc
JAK_loc_cbds_osm_2025_utility_150m.asc
JAK_loc_health_osm_2025_utility_150m.asc
JAK_loc_markets_osm_2025_utility_150m.asc
JAK_loc_malls_osm_2025_utility_150m.asc
JAK_loc_schools_osm_2025_utility_150m.asc
JAK_loc_suburban_centres_osm_2025_utility_150m.asc
JAK_loc_attractive_areas_osm_2025_utility_150m.asc
JAK_loc_attractive_areas_osm_2025_utility_150m.asc
JAK_hyd_water_utility_150m.asc
JAK_TRA_roads_OSM_2025_utility_150m.asc
JAK_URB_urbanisation_2015_GHSL_utility_150m.asc
JAK_URB_urbanisation_2025_GHSL_utility_150m.asc
JAK_POP_density_normal_2000_WDPop_150m.asc
JAK_POP_density_normal_2020_WDPop_150m.asc
JAK_ECO_real_estate_reclass_150m.asc
JAK_TRA_roads_OSM_2025_major_150m.asc
JAK_URB_urbanisation_2015_GHSL_150m.asc
JAK_URB_urb

In [21]:
# code adapted from https://www.geeksforgeeks.org/python-shutil-copyfile-method/

success_list = []
fail_list = []
for key in renaming_dict.keys():
    source = ascii150_path / renaming_dict[key]
    dest = ti_city_ascii_path / key
    
    try :
        shutil.copyfile(source, dest)
        success_list.append(dest)
    
    # If Source is a file but destination is a directory
    except IsADirectoryError:
        print("Source is a file but destination is a directory.")
        fail_list.append(dest)
    
    # If source and destination are same
    except shutil.SameFileError:
        print("Source and destination represents the same file.")
        fail_list.append(dest)

    # If source is a directory but destination is a file
    except NotADirectoryError:
        print("Source is a directory but destination is a file.")
        fail_list.append(dest)
    
    # For permission related errors
    except PermissionError:
        print("Operation not permitted.")
        fail_list.append(dest)
    
    # For other errors
    except:
        print("Error occurred while copying file:\n", source)
        fail_list.append(dest)
print('Tried to copy:', len(renaming_dict), 'files\n', 'Succesfully copied:', len(success_list), 'files\n','The files copied were:',)
print('\n'.join(''.join(str(sl)) for sl in success_list))
if len(fail_list) > 0:
    print('Failed to copy:', len(fail_list), 'files\n','The files not copied were:')
    print('\n'.join(''.join(str(sl)) for sl in fail_list))

Tried to copy: 27 files
 Succesfully copied: 27 files
 The files copied were:
..\model\JAK\data\in\slope.asc
..\model\JAK\data\in\exclusion.asc
..\model\JAK\data\in\districts.asc
..\model\JAK\data\in\structureplan.asc
..\model\JAK\data\in\airport.asc
..\model\JAK\data\in\cbd.asc
..\model\JAK\data\in\healthfacilities.asc
..\model\JAK\data\in\markets.asc
..\model\JAK\data\in\shoppingmalls.asc
..\model\JAK\data\in\schools.asc
..\model\JAK\data\in\suburban.asc
..\model\JAK\data\in\attractive_early.asc
..\model\JAK\data\in\attractive_late.asc
..\model\JAK\data\in\water.asc
..\model\JAK\data\in\roads.asc
..\model\JAK\data\in\bu_early_dist.asc
..\model\JAK\data\in\bu_late_dist.asc
..\model\JAK\data\in\density_early.asc
..\model\JAK\data\in\density_late.asc
..\model\JAK\data\in\landvalue.asc
..\model\JAK\data\in\road-presence.asc
..\model\JAK\data\in\urban_early.asc
..\model\JAK\data\in\urban_late.asc
..\model\JAK\data\in\exclusion_ssp1.asc
..\model\JAK\data\in\exclusion_ssp2.asc
..\model\JAK\

# The end (for now).